# 고정 Gateway IP: AgentCore Gateway 트래픽 허용 목록 구성

이 실습에서는 외부 서비스가 허용 목록에 추가할 수 있도록 Amazon Bedrock AgentCore Gateway에 **알려진 고정 IP 주소**를 할당하는 방법을 알아봅니다. AgentCore Gateway가 외부 엔드포인트에 연결할 때 트래픽은 변경될 수 있는 AWS 관리형 IP 주소에서 시작됩니다. 

이로 인해 다음과 같은 문제가 발생합니다.
- **API Gateway의 WAF 규칙**: 퍼블릭 API Gateway가 WAF 뒤에 있어 AgentCore Gateway의 도구 호출이 차단됩니다. WAF 허용 목록에 추가할 특정 IP가 필요하지만 AgentCore Gateway의 소스 IP 범위는 고정되어 있지 않으므로, 동적으로 변경되는 IP 집합의 WAF 허용 목록을 유지하는 것은 현실적이지 않습니다.
- **MCP 서버 IP 허용 목록**: MCP 서버 제공업체에서 허용 목록에 추가할 고정 소스 IP를 요구하지만, AgentCore Gateway의 소스 IP는 동적입니다.

## 해결 방법

VPC 송신을 사용하여 AgentCore Gateway 트래픽을 VPC로 라우팅하고 **Elastic IP가 연결된 NAT Gateway**를 통해 외부로 전송합니다. MCP 서버에는 모든 요청이 NAT Gateway의 고정 Elastic IP에서 오는 것으로 표시되며, 이 IP를 MCP 서버 운영자에게 제공하여 허용 목록에 추가할 수 있습니다.

![고정 Gateway IP 아키텍처](images/gateway-static-ip.png)


## 작동 방식

1. AgentCore Gateway는 **managed VPC resource**를 사용하여 Resource Gateway를 통해 트래픽을 VPC로 라우팅합니다.
2. Resource Gateway ENI는 송신 트래픽(0.0.0.0/0)을 NAT Gateway를 통해 라우팅하는 **프라이빗 서브넷**에 배치됩니다.
3. NAT Gateway에는 고정 퍼블릭 IP 주소인 **Elastic IP**가 연결됩니다.
4. 외부 MCP 서버로 향하는 모든 트래픽은 이 단일 Elastic IP를 통해 나갑니다.
5. MCP 서버가 이 IP를 허용 목록에 추가하면 AgentCore Gateway 트래픽만 허용됩니다.

> **복원력과 단순성:** 이 실습에서는 단일 고정 IP를 위해 하나의 NAT Gateway를 사용합니다. 프로덕션 환경에서는 고가용성을 위해 가용 영역별로 NAT Gateway 하나를 배포하는 방안을 고려하세요. 각 NAT Gateway에는 자체 Elastic IP가 있으므로 허용 목록에 추가할 여러 IP를 MCP 서버에 제공해야 합니다.

VPC 송신 및 managed VPC resource에 대한 배경 정보는 [프로젝트 README](../README.md)와 [고급 개념 README](./README.md)를 참조하세요.

## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) 완료(VPC + AgentCore Gateway 배포)

## 1단계: 종속성 설치 및 라이브러리 가져오기

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0에서 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")
ec2_client = session.client("ec2")

# Cognito 클라이언트 보안 암호 가져오기
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

## 2단계: 고정 IP 확인

[실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb)에서 배포한 VPC에는 가용 영역별로 하나씩 Elastic IP가 연결된 NAT Gateway가 이미 있습니다. 각 프라이빗 서브넷은 해당 가용 영역의 NAT Gateway를 통해 송신 트래픽(0.0.0.0/0)을 라우팅합니다.

**단일 고정 IP**를 사용하기 위해 프라이빗 서브넷 하나를 사용합니다. 해당 서브넷의 모든 Resource Gateway ENI는 하나의 Elastic IP가 연결된 NAT Gateway 하나를 통해 외부로 나갑니다.

> **참고:** 추가 인프라는 필요하지 않습니다. 실습 0의 기존 VPC, 프라이빗 서브넷 및 NAT Gateway를 재사용합니다.

In [ ]:
# 첫 번째 프라이빗 서브넷 사용(단일 AZ = 단일 NAT Gateway = 단일 고정 IP)
STATIC_IP_SUBNET_ID = VPC_USW2_PRIVATE_SUBNETS[0]

# 이 서브넷이 트래픽을 라우팅하는 NAT Gateway 확인
route_tables = ec2_client.describe_route_tables(
    Filters=[{"Name": "association.subnet-id", "Values": [STATIC_IP_SUBNET_ID]}]
)["RouteTables"]

nat_gw_id = None
for route in route_tables[0]["Routes"]:
    if route.get("DestinationCidrBlock") == "0.0.0.0/0" and route.get("NatGatewayId"):
        nat_gw_id = route["NatGatewayId"]
        break

# NAT Gateway의 Elastic IP 가져오기
nat_gw = ec2_client.describe_nat_gateways(NatGatewayIds=[nat_gw_id])["NatGateways"][0]
STATIC_IP = nat_gw["NatGatewayAddresses"][0]["PublicIp"]
AZ = nat_gw["SubnetId"]  # NAT GW 서브넷(퍼블릭)

# 프라이빗 서브넷의 AZ 가져오기
subnet_info = ec2_client.describe_subnets(SubnetIds=[STATIC_IP_SUBNET_ID])["Subnets"][0]
AZ = subnet_info["AvailabilityZone"]

print(f"Private subnet:  {STATIC_IP_SUBNET_ID} ({AZ})")
print(f"NAT Gateway:     {nat_gw_id}")
print(f"Static IP:       {STATIC_IP}")
print(f"\nAll outbound traffic from this subnet exits via {STATIC_IP}")
print("Provide this IP to your MCP server operator for allowlisting.")

In [ ]:
# Resource Gateway ENI용 보안 그룹 생성
# 새 보안 그룹은 기본적으로 모든 송신 트래픽 허용
sg = ec2_client.create_security_group(
    GroupName=f"static-ip-rg-sg-{int(time.time())}",
    Description="Resource Gateway SG for static IP lab - allows outbound HTTPS",
    VpcId=VPC_USW2_ID,
)
STATIC_IP_SG_ID = sg["GroupId"]
print(f"Security group: {STATIC_IP_SG_ID}")

## 3단계: AgentCore Gateway 대상 생성

퍼블릭 HTTP 테스트 서비스인 [httpbin.org](https://httpbin.org)을 가리키는 Gateway 대상을 생성합니다. `/ip` 엔드포인트는 호출자의 소스 IP 주소를 반환하므로 트래픽이 고정 Elastic IP를 통해 나가는지 확인할 수 있습니다.

Resource Gateway를 전용 프라이빗 서브넷에 배치하면 httpbin.org로 향하는 모든 트래픽이 다음 경로를 통과합니다.

```
AgentCore Gateway -> VPC Lattice -> Resource Gateway ENI (private subnet) -> NAT Gateway (EIP) -> httpbin.org
```

In [ ]:
# httpbin.org의 /ip 엔드포인트에 대한 OpenAPI 스키마 정의
openapi_schema = {
    "openapi": "3.0.0",
    "info": {"title": "IP Check", "version": "1.0.0"},
    "servers": [{"url": "https://httpbin.org"}],
    "paths": {
        "/ip": {
            "get": {
                "operationId": "getOriginIp",
                "summary": "Returns the origin IP address of the request",
                "responses": {
                    "200": {
                        "description": "Origin IP",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "type": "object",
                                    "properties": {"origin": {"type": "string"}},
                                }
                            }
                        },
                    }
                },
            }
        }
    },
}

OPENAPI_SCHEMA = json.dumps(openapi_schema)
print("Target endpoint: https://httpbin.org/ip")
print(f"Expected source IP: {STATIC_IP}")

In [ ]:
# 더미 API 키 자격 증명 공급자 생성(AgentCore Gateway에 필요)
# httpbin.org은 이 헤더를 무시하지만 API에는 credentialProviderConfigurations가 필요함
cred_response = agentcore.create_api_key_credential_provider(
    name="static-ip-check-api-key",
    apiKey="dummy-key-for-httpbin",
)
CRED_PROVIDER_ARN = cred_response["credentialProviderArn"]
print(f"Credential provider ARN: {CRED_PROVIDER_ARN}")

response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="static-ip-check",
    description="IP check via httpbin - verifies static IP egress through NAT Gateway",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": OPENAPI_SCHEMA,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": CRED_PROVIDER_ARN,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
    privateEndpoint={
        "managedVpcResource": {
            "vpcIdentifier": VPC_USW2_ID,
            "subnetIds": [STATIC_IP_SUBNET_ID],
            "endpointIpAddressType": "IPV4",
            "securityGroupIds": [STATIC_IP_SG_ID],
        }
    },
)

TARGET_ID = response["targetId"]
print(f"Target ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Managed resources: {target.get('privateEndpoint', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 4단계: 고정 IP 확인

AgentCore Gateway를 통해 httpbin `/ip` 엔드포인트를 호출합니다. 응답에는 httpbin에 표시되는 소스 IP가 포함되며, 이 IP는 NAT Gateway의 Elastic IP와 일치해야 합니다.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

# AgentCore Gateway를 통해 IP 확인 엔드포인트 호출
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "static-ip-check___getOriginIp", "arguments": {}},
        "id": 1,
    },
)

result = response.json()
print("Response:")
print(json.dumps(result, indent=2))

In [ ]:
# 응답에서 원본 IP를 추출하여 Elastic IP와 비교
try:
    tool_result = result["result"]["content"][0]["text"]
    origin_data = json.loads(tool_result)
    origin_ip = origin_data["origin"]
except (KeyError, json.JSONDecodeError, IndexError):
    origin_ip = "(could not parse)"
    print(f"Raw result: {result}")

print(f"Origin IP seen by httpbin:  {origin_ip}")
print(f"NAT Gateway Elastic IP:     {STATIC_IP}")
print()
if origin_ip == STATIC_IP:
    print("MATCH - All AgentCore Gateway traffic exits through the static Elastic IP.")
    print(f"Provide this IP to your MCP server operator for allowlisting: {STATIC_IP}")
else:
    print("MISMATCH - Traffic may be using a different egress path.")
    print("Check the subnet route table and NAT Gateway configuration.")

## 5단계: 실제 MCP 서버에 연결

고정 IP를 확인했으므로 이제 실제 외부 MCP 서버인 [Exa](https://exa.ai)에 연결합니다. Exa는 `https://mcp.exa.ai/mcp`에서 MCP 엔드포인트를 제공하는 검색 API입니다.

이 과정은 AgentCore Gateway 트래픽을 고정 IP를 통해 외부 MCP 서버로 라우팅하여 MCP 서버 운영자가 해당 IP를 허용 목록에 추가할 수 있도록 하는 실제 사용 사례를 보여 줍니다.

In [ ]:
# 예시 MCP 서버용 Gateway 대상 생성

exa_response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="exa-mcp",
    description="Exa search MCP server via static IP egress through NAT Gateway",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": "https://mcp.exa.ai/mcp",
            }
        }
    },
    privateEndpoint={
        "managedVpcResource": {
            "vpcIdentifier": VPC_USW2_ID,
            "subnetIds": [STATIC_IP_SUBNET_ID],
            "endpointIpAddressType": "IPV4",
            "securityGroupIds": [STATIC_IP_SG_ID],
        }
    },
)

EXA_TARGET_ID = exa_response["targetId"]
print(f"Exa Target ID: {EXA_TARGET_ID}")
print(f"Status:        {exa_response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=EXA_TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nExa target is active!")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

In [ ]:
# Exa MCP 서버에서 사용할 수 있는 도구 나열
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
# Exa 도구만 필터링
tools = response.json().get("result", {}).get("tools", [])
exa_tools = [t for t in tools if t["name"].startswith("exa-mcp___")]
print(f"Exa MCP tools ({len(exa_tools)}):")
for t in exa_tools:
    print(f"  {t['name']}: {t.get('description', '')[:80]}")

In [ ]:
# AgentCore Gateway를 통해 Exa를 사용하여 웹 검색
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "exa-mcp___web_search_exa",
            "arguments": {"query": "Amazon Bedrock AgentCore Gateway"},
        },
        "id": 2,
    },
)
print("Exa search results:")
print(json.dumps(response.json(), indent=2))

## 프로덕션 고려 사항

### 고가용성

이 실습에서는 구성을 단순화하기 위해 하나의 가용 영역에 **단일 NAT Gateway**를 사용합니다. NAT Gateway가 하나이면 허용 목록에 추가할 Elastic IP도 하나입니다. 프로덕션 환경에서는 복원력을 위해 AZ별로 NAT Gateway 하나를 배포하는 방안을 고려하세요.


NAT Gateway가 여러 개이면 Elastic IP도 여러 개입니다. 허용 목록에 추가할 수 있도록 모든 IP를 MCP 서버 운영자에게 제공하세요. 한 AZ에 장애가 발생하면 트래픽이 다른 AZ의 NAT Gateway로 장애 조치되며 계속 알려진 IP를 사용합니다.


### 전용 송신 서브넷

더 세밀하게 제어하려면 Resource Gateway ENI 전용 **프라이빗 서브넷**을 생성하세요. 이 서브넷에는 특정 NAT Gateway를 가리키는 자체 라우팅 테이블이 있으므로 AgentCore Gateway 송신 트래픽을 VPC의 다른 워크로드와 격리할 수 있습니다.


장점:
- **영향 범위**: 보안 그룹 및 NACL 규칙의 범위가 송신 서브넷으로만 제한됩니다.
- **감사 가능성**: 송신 서브넷의 VPC Flow Logs가 AgentCore Gateway 트래픽만 캡처합니다.
- **비용 귀속**: NAT Gateway 데이터 처리 요금이 Gateway 송신 트래픽으로 분리됩니다.

### 비용

| 리소스 | 비용 |
|----------|------|
| Elastic IP(연결됨) | 요금 없음 |
| NAT Gateway | 시간당 약 \$0.045 + 처리된 GB당 \$0.045 |
| VPC Lattice | 데이터 처리 요금(GB당) |

### 보안

고정 IP 방식은 VPC의 NAT Gateway를 통해 라우팅된 트래픽만 MCP 서버에 도달하도록 보장합니다. 조직의 다른 영역에서 발생하는 트래픽은 소스 IP가 다르므로 직접 액세스가 차단됩니다. 심층 방어를 위해 이 방식을 MCP 서버 자체 인증과 함께 사용하세요.


### VPC Block Public Access(BPA)

[VPC Block Public Access](https://docs.aws.amazon.com/vpc/latest/userguide/security-vpc-bpa.html)를 사용하면 VPC 전체에서 Internet Gateway를 통과하는 모든 트래픽을 차단할 수 있습니다. **NAT Gateway는 VPC BPA를 우회합니다.** BPA의 Internet Gateway 제한으로 제어되지 않는 별도의 송신 경로를 사용하기 때문입니다.

따라서 VPC BPA는 고정 IP 패턴을 효과적으로 보완합니다.

- VPC에서 **양방향 BPA 활성화**: IGW를 통한 모든 직접 인터넷 액세스를 차단합니다.
- **NAT Gateway 트래픽은 영향을 받지 않음**: 프라이빗 서브넷의 Resource Gateway ENI는 NAT Gateway를 통해 계속 송신 트래픽을 라우팅합니다.
- **심층 방어**: VPC의 리소스에 퍼블릭 IP가 잘못 구성되어 있더라도 VPC BPA가 해당 리소스의 직접 인터넷 연결을 차단합니다.

이렇게 하면 NAT Gateway의 Elastic IP가 VPC의 **유일한** 송신 경로가 되어 송신 트래픽을 완전히 제어할 수 있습니다.

## 정리

1. Gateway 대상 삭제
2. 보안 그룹 삭제

> **참고:** AgentCore의 관리형 Resource Gateway ENI가 보안 그룹을 계속 참조하는 경우 `DependencyViolation`이 발생할 수 있습니다. 대상을 삭제한 후 몇 분 정도 기다렸다가 다시 시도하세요.

In [ ]:
# # 1단계: Gateway target 삭제
# for tid, name in [(TARGET_ID, "static-ip-check"), (EXA_TARGET_ID, "exa-mcp")]:
#     agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=tid)
#     print(f"Deleting target: {tid} ({name})")
#     while True:
#         try:
#             t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=tid)
#             print(f"  Status: {t['status']}")
#             time.sleep(15)
#         except agentcore.exceptions.ResourceNotFoundException:
#             print(f"  {name} deleted.")
#             break

# # credential provider 삭제
# agentcore.delete_api_key_credential_provider(name="static-ip-check-api-key")
# agentcore.delete_api_key_credential_provider(name="exa-mcp-api-key")
# print("Deleted credential providers")

In [ ]:
# # 2단계: security group 삭제
# # "DependencyViolation"으로 실패하면 ENI가 해제될 때까지 몇 분 기다리세요.
# try:
#     ec2_client.delete_security_group(GroupId=STATIC_IP_SG_ID)
#     print(f"Deleted security group: {STATIC_IP_SG_ID}")
# except ec2_client.exceptions.ClientError as e:
#     if "DependencyViolation" in str(e):
#         print(
#             f"SG {STATIC_IP_SG_ID} still has dependencies. Wait a few minutes and retry."
#         )
#     else:
#         raise